# Рекурентні нейронні мережі

У попередньому модулі ми розглянули багаті семантичні представлення тексту. Архітектура, яку ми використовували, охоплює агреговане значення слів у реченні, але не враховує **порядок** слів, оскільки операція агрегації, яка слідує за вбудовуваннями, вилучає цю інформацію з оригінального тексту. Оскільки ці моделі не можуть представляти порядок слів, вони не можуть розв’язувати більш складні або неоднозначні завдання, такі як генерація тексту або відповіді на запитання.

Щоб врахувати значення послідовності тексту, ми використовуватимемо нейронну мережеву архітектуру, яка називається **рекурентна нейронна мережа** або RNN. При використанні RNN ми передаємо наше речення через мережу по одному токену за раз, і мережа виробляє деякий **стан**, який ми потім передаємо мережі знову разом із наступним токеном.

![Зображення, що показує приклад генерації рекурентної нейронної мережі.](../../../../../translated_images/uk/rnn.27f5c29c53d727b5.webp)

Якщо задано вхідну послідовність токенів $X_0,\dots,X_n$, RNN створює послідовність блоків нейронної мережі і тренує цю послідовність кінцево-в-кінці за допомогою зворотного поширення помилки. Кожен мережевий блок приймає пару $(X_i,S_i)$ на вхід і повертає результат $S_{i+1}$. Кінцевий стан $S_n$ або вихід $Y_n$ подається у лінійний класифікатор для отримання результату. Всі мережеві блоки використовують однакові ваги та тренуються кінцево-в-кінці одним проходом зворотного поширення.

> На рисунку зверху показана рекурентна нейронна мережа у розгорнутій формі (зліва) та в більш компактному рекурентному поданні (справа). Важливо усвідомити, що всі RNN-ядра мають однакові **розподільні ваги**.

Оскільки вектори стану $S_0,\dots,S_n$ проходять через мережу, RNN здатна вивчати послідовні залежності між словами. Наприклад, коли слово *not* з’являється десь у послідовності, вона може навчитися заперечувати певні елементи всередині вектора стану.

Всередині кожна RNN-клітина містить дві матриці ваг: $W_H$ та $W_I$, а також зсув $b$. На кожному кроці RNN, маючи вхідний $X_i$ та вхідний стан $S_i$, вихідний стан обчислюється як $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, де $f$ — функція активації (часто $\tanh$).

> Для задач, таких як генерація тексту (які ми розглянемо у наступному уроці) або машинний переклад, ми також хочемо отримувати деяке вихідне значення на кожному кроці RNN. У цьому випадку є також додаткова матриця $W_O$, а вихід обчислюється як $Y_i=f(W_O\times S_i+b_O)$.

Давайте подивимось, як рекурентні нейронні мережі можуть допомогти нам класифікувати наш набір новин.

> Для середовища пісочниці нам потрібно виконати наступну клітинку, щоб переконатися, що потрібна бібліотека встановлена, а дані попередньо завантажені. Якщо ви працюєте локально, ви можете пропустити цю клітинку.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Під час навчання великих моделей розподіл пам’яті GPU може стати проблемою. Ми також можемо експериментувати з різними розмірами мінібатчів, щоб дані вміщувалися у пам’ять GPU, і тренування було достатньо швидким. Якщо ви запускаєте цей код на власній машині з GPU, ви можете експериментувати з регулюванням розміру мінібатча для прискорення навчання.

> **Примітка**: Відомо, що деякі версії драйверів NVidia не звільняють пам’ять після навчання моделі. Ми запускаємо кілька прикладів у цьому ноутбуці, і це може спричинити вичерпання пам’яті в певних конфігураціях, особливо якщо ви проводите власні експерименти в межах того ж ноутбука. Якщо під час початку навчання моделі виникають дивні помилки, можливо, варто перезавантажити ядро ноутбука.


In [3]:
batch_size = 16
embed_size = 64

## Простий RNN класифікатор

У випадку простого RNN кожен рекурентний блок є простою лінійною мережею, яка приймає вхідний вектор та вектор стану і видає новий вектор стану. У Keras це можна представити шаром `SimpleRNN`.

Хоч ми можемо подавати one-hot закодовані токени безпосередньо до шару RNN, це не є хорошою ідеєю через їх високу розмірність. Тому ми використаємо шар embedding для зменшення розмірності векторів слів, після якого буде шар RNN, і нарешті класифікатор `Dense`.

> **Примітка**: У випадках, коли розмірність не така велика, наприклад при токенізації на рівні символів, може мати сенс подавати one-hot закодовані токени безпосередньо у RNN клітинку.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Примітка:** Тут ми використовуємо нетренований шар для векторного подання для простоти, але для кращих результатів можна використовувати попередньо навчену модель векторного подання з Word2Vec, як було описано в попередньому уроці. Було б корисним для вас адаптувати цей код для роботи з попередньо навченими векторними поданнями.

Тепер давайте навчимо нашу RNN. Взагалі RNN досить складно навчати, тому що після розгортання клітинок RNN по довжині послідовності загальна кількість шарів, за якими виконується зворотнє поширення, стає дуже великою. Тому потрібно вибрати меншу швидкість навчання і навчати мережу на більшій кількості даних, щоб отримати хороші результати. Це може зайняти досить багато часу, тому перевага віддається використанню GPU.

Щоб прискорити процес, ми будемо навчати модель RNN тільки на заголовках новин, пропускаючи опис. Ви можете спробувати навчати модель з описом і подивитися, чи вдасться вам навчити модель.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Зауважте**, що точність тут, ймовірно, буде нижчою, оскільки ми навчаємося лише на заголовках новин.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Тепер, коли ми використовуємо маскування, ми можемо навчати модель на всьому наборі даних заголовків та описів.

> **Примітка**: Ви помітили, що ми використовували векторизатор, навчений на заголовках новин, а не на всьому тексті статті? Це потенційно може призводити до ігнорування деяких токенів, тому краще повторно навчити векторизатор. Однак, це може мати дуже незначний ефект, тож для простоти ми залишимо попередній попередньо навченний векторизатор.


## LSTM: Довга короткочасна пам’ять

Однією з основних проблем RNN є **затухання градієнтів**. RNN можуть бути досить довгими, і їм може бути важко поширювати градієнти аж до першого шару мережі під час зворотного поширення помилки. Коли це трапляється, мережа не може вивчати залежності між віддаленими токенами. Один із способів уникнути цієї проблеми — ввести **явне керування станом** за допомогою **воріт**. Дві найпоширеніші архітектури, що вводять ворота, — це **довга короткочасна пам’ять** (LSTM) та **ворітний релейний блок** (GRU). Тут ми розглянемо LSTM.

![Зображення, що показує приклад комірки довгої короткочасної пам’яті](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Мережа LSTM організована подібно до RNN, але існують два стани, які передаються з шару в шар: власне стан $c$ та прихований вектор $h$. На кожному вузлі прихований вектор $h_{t-1}$ поєднується з вхідним $x_t$, і разом вони контролюють, що буде зі станом $c_t$ та виходом $h_{t}$ через **ворота**. Кожні ворота мають сигмоїдальну активацію (вихід у діапазоні $[0,1]$), яку можна уявляти як побітову маску при множенні на вектор стану. У LSTM є наступні ворота (зліва направо на наведеному вище малюнку):
* **ворота забуття**, які визначають, які компоненти вектора $c_{t-1}$ потрібно забути, а які пропустити далі.
* **вхідні ворота**, які визначають, скільки інформації з вхідного вектора та попереднього прихованого вектора потрібно включити у вектор стану.
* **вихідні ворота**, які приймають новий вектор стану і визначають, які його компоненти будуть використані для створення нового прихованого вектора $h_t$.

Компоненти стану $c$ можна уявляти як прапорці, які можна вмикати та вимикати. Наприклад, коли ми зустрічаємо ім’я *Alice* у послідовності, ми припускаємо, що це жінка, і піднімаємо прапорець у стані, який говорить, що у реченні є іменник жіночого роду. Коли ми далі зустрічаємо слова *and Tom*, ми піднімаємо прапорець, що у реченні є множина. Таким чином, маніпулюючи станом, ми можемо відстежувати граматичні властивості речення.

> **Примітка**: Ось чудовий ресурс для розуміння внутрішньої структури LSTM: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) Крістофера Олаха.

Хоча внутрішня структура комірки LSTM може виглядати складною, Keras приховує цю реалізацію всередині шару `LSTM`, тому єдине, що нам потрібно зробити в наведеному вище прикладі, це замінити рекурентний шар:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Примітка**: навчання LSTM також досить повільне, і на початку тренування ви можете не побачити значного збільшення точності. Вам, можливо, доведеться продовжувати тренування деякий час, щоб досягти хорошої точності.


## Двонапрямні та багатошарові RNN

У наших прикладах досі рекурентні мережі працюють від початку послідовності до кінця. Це здається нам природним, тому що це відповідає тому ж напрямку, у якому ми читаємо або слухаємо мову. Однак для сценаріїв, що потребують випадкового доступу до вхідної послідовності, логічніше запускати рекурентне обчислення в обох напрямках. RNN, які дозволяють обчислення в обох напрямках, називаються **двонапрямними** RNN, і їх можна створити, обернувши рекурентний шар спеціальним шаром `Bidirectonal`.

> **Примітка**: Шар `Bidirectional` створює дві копії шару всередині себе і встановлює властивість `go_backwards` однієї з цих копій у `True`, змушуючи її рухатися у протилежному напрямку послідовності.

Рекурентні мережі, однонапрямні чи двонапрямні, захоплюють закономірності у послідовності й зберігають їх у векторах стану або повертають як вихідні дані. Як і у випадку з згортковими мережами, ми можемо побудувати ще один рекурентний шар після першого, щоб вловлювати більш високорівневі закономірності, побудовані з нижчорівневих, які видобуває перший шар. Це приводить нас до поняття **багатошарової RNN**, яка складається з двох або більше рекурентних мереж, де вихід попереднього шару подається на вхід наступного.

![Зображення, що показує багатошарову long-short-term-memory-RNN](../../../../../translated_images/uk/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Зображення з [цього чудового допису](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) Фернандо Лопеса.*

Keras робить побудову цих мереж простою задачею, бо вам просто потрібно додати більше рекурентних шарів до моделі. Для всіх шарів, крім останнього, потрібно вказати параметр `return_sequences=True`, тому що нам потрібно, щоб шар повертав усі проміжні стани, а не тільки кінцевий стан рекурентного обчислення.

Давайте побудуємо двошарову двонапрямну LSTM для нашої задачі класифікації.

> **Примітка**: цей код знову виконується досить довго, але дає найвищу точність, яку ми бачили досі. Тож, можливо, варто почекати і подивитися на результат.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN для інших завдань

До цього часу ми зосереджувалися на використанні RNN для класифікації послідовностей тексту. Але вони можуть виконувати набагато більше завдань, таких як генерація тексту та машинний переклад — ці завдання ми розглянемо у наступному розділі.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
